# 유담 인식 모델 학습 ②③④ (Colab T4)

| 단계 | 학습 데이터 | 설정 | 상태 |
|---|---|---|---|
| ② 파인튜닝 | K-Art train | 마지막 4블록 + projection, SupCon, 300스텝, lr 1e-5, 시드 0 | 완료 (시연 모델) |
| ③ + 증강 | ②와 같음 | ② + albumentations 증강 | 완료 |
| ④ + A/B hard negative | ② + **A01·B01 gallery 54장** | ② + 매 배치에 A01↔B01 쌍 | **이번에 실행** |

**준비물**: PC의 `data/` 폴더에 있는 zip을 Google Drive에 올린다 (폴더는 어디든 된다. 1번 셀이 찾아 준다)
- `yudam_colab.zip` — K-Art train/val + manifest + 학습 코드 (②③④ 모두 필요)
- `yudam_colab_ab.zip` — A01·B01 gallery + artifacts.db + ④ 지원 학습 코드 + hard_pairs_ab.csv (④에만 필요)

**실행**: 런타임 → 런타임 유형 변경 → **T4 GPU** → 1번 셀부터 차례로. 필요한 단계 섹션만 실행하면 된다.
런타임이 재시작되면 `/content` 파일이 사라지므로 1번 셀부터 다시 실행한다.

**규칙**
- test는 올리지 않는다. gallery는 ④에서 A01·B01만 예외로 학습에 쓴다 (A·B는 K-Art train에 없어 차이를 배울 사진이 gallery뿐).
- 단계마다 달라지는 건 학습 방법 하나뿐이다. 설정(스텝, lr, 시드, 배치)을 바꾸지 않는다.
- A·B는 val이 없어 ④ 설정을 고를 수 없다. ④는 **한 번만** 학습한다.
- Drive에 이미 있는 체크포인트는 덮어쓰지 않는다 (비교표 기준 모델). 다시 돌리면 `_rerun` 이름으로 저장된다.
- AI Hub 약관: 데이터 원본·가공본 공유 금지 → Drive 폴더 링크를 공유하지 말 것.

## 1. 준비 (GPU 확인 → Drive 연결 → zip 찾아서 풀기 → 점검)

In [ ]:
import glob, os, shutil, subprocess, torch

assert torch.cuda.is_available(), "GPU가 없어요: 런타임 → 런타임 유형 변경 → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

def find_zip(name, required=True):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    if not hits:
        msg = (f"Drive에서 {name}을 못 찾았어요. PC의 data/{name}을 Drive에 올려 주세요 "
               f"(마운트한 계정이 zip을 올린 계정과 같은지도 확인)")
        if required:
            raise FileNotFoundError(msg)
        print("[없음]", msg)
        return None
    print(f"찾음: {hits[0]}" + (f"  (같은 이름 {len(hits)}개 중 첫 번째)" if len(hits) > 1 else ""))
    return hits[0]

ZIP_BASE = find_zip('yudam_colab.zip')
ZIP_AB = find_zip('yudam_colab_ab.zip', required=False)  # ④에만 필요
OUT_DRIVE = os.path.join(os.path.dirname(ZIP_BASE), 'out')

# 순서 중요: 기존 zip 먼저, ④ zip을 나중에 풀어서 train_metric.py를 ④ 지원 버전으로 덮어쓴다
# (④ 지원 버전은 ②③도 그대로 돌린다)
os.makedirs('/content/yudam', exist_ok=True)
import zipfile
for z in (ZIP_BASE, ZIP_AB):
    if z:
        # unzip 명령은 한글 파일명을 깨뜨릴 수 있어서 파이썬으로 푼다
        with zipfile.ZipFile(z) as zf:
            zf.extractall('/content/yudam')

code_text = open('/content/yudam/train/train_metric.py', encoding='utf-8').read()
ok = {
    'manifest': os.path.exists('/content/yudam/train/manifest.csv'),
    'train 폴더': os.path.isdir('/content/yudam/data/train'),
    'val 폴더': os.path.isdir('/content/yudam/data/val'),
    'test 없음': not os.path.exists('/content/yudam/data/test'),
}
for k, v in ok.items():
    print(f"{k}: {v}")
assert all(ok.values()), "위에서 False인 항목을 확인해 주세요"

def gallery_files_openable():
    """DB에 적힌 A01·B01 gallery 경로 54개가 실제로 열리는지 (한글 파일명 깨짐 점검)"""
    import sqlite3
    con = sqlite3.connect('/content/yudam/data/artifacts.db')
    rows = con.execute("SELECT path FROM artifact_images WHERE split='gallery' AND artifact_id IN ('A01','B01')").fetchall()
    missing = [p for (p,) in rows if not os.path.exists(os.path.join('/content/yudam/data', p))]
    print(f"gallery 파일 {len(rows) - len(missing)}/{len(rows)}개 열림", ("" if not missing else f"| 없는 파일 예: {missing[:2]}"))
    return len(rows) == 54 and not missing

AB_READY = bool(ZIP_AB) and gallery_files_openable() and all([
    os.path.exists('/content/yudam/data/artifacts.db'),
    len(glob.glob('/content/yudam/data/gallery/A01/*')) == 34,
    len(glob.glob('/content/yudam/data/gallery/B01/*')) == 20,
    os.path.exists('/content/yudam/train/hard_pairs_ab.csv'),
    '--gallery-ids' in code_text,
])
print("④ 준비:", AB_READY, "(A01 34장, B01 20장, DB, hard_pairs, ④ 지원 코드)")
print()
print("결과 저장 위치:", OUT_DRIVE)

def save_ckpt(stage):
    """Drive에 저장. 이미 있으면 덮어쓰지 않고 _rerun 이름으로 저장한다."""
    src = f'/content/out/ckpt_stage{stage}.pt'
    assert os.path.exists(src), "학습이 끝나지 않았어요. 위 출력의 마지막 오류를 확인해 주세요"
    os.makedirs(OUT_DRIVE, exist_ok=True)
    dst = os.path.join(OUT_DRIVE, f'ckpt_stage{stage}.pt')
    if os.path.exists(dst):
        dst = os.path.join(OUT_DRIVE, f'ckpt_stage{stage}_rerun.pt')
        print("[주의] 기존 체크포인트가 있어 덮어쓰지 않고 _rerun 이름으로 저장합니다")
    shutil.copy(src, dst)
    print("저장 완료:", dst, f"{os.path.getsize(src) / 1e6:.0f}MB")

In [ ]:
!pip -q install open_clip_torch albumentations

## 2. 동작 점검 (3스텝, 1~2분)
오류 없이 끝나면 필요한 단계로 간다. `Warning: You are sending unauthenticated requests to the HF Hub` 경고는 무시해도 된다.

In [ ]:
%cd /content/yudam
if AB_READY:
    !python train/train_metric.py --stage 4 --manifest train/manifest.csv --data-root data --out-dir /content/out_check --steps 3 --gallery-ids A01,B01 --hard-pairs train/hard_pairs_ab.csv
else:
    !python train/train_metric.py --stage 2 --manifest train/manifest.csv --data-root data --out-dir /content/out_check --steps 3

## 3. ② 파인튜닝 (증강 없음) — 이미 완료, 다시 돌릴 필요 없음
출력에서 `device=cuda`, 학습 전후 `val Top-1` 두 줄을 확인한다.

In [ ]:
%cd /content/yudam
!python train/train_metric.py --stage 2 --manifest train/manifest.csv --data-root data --out-dir /content/out --steps 300
save_ckpt(2)

## 4. ③ + 증강 (②와 같은 설정, 증강만 추가) — 이미 완료, 다시 돌릴 필요 없음

In [ ]:
%cd /content/yudam
!python train/train_metric.py --stage 3 --manifest train/manifest.csv --data-root data --out-dir /content/out --steps 300
save_ckpt(3)

## 5. ④ ② + A/B hard negative — **이번에 실행** (T4 기준 수 분)
②와 같은 설정에 A01·B01 gallery 54장을 학습에 넣고, 매 배치에 A01↔B01 쌍을 함께 넣는다.
출력 첫 줄에 `gallery에서 추가: A01 34장, B01 20장`, 다음 줄에 `device=cuda`가 나와야 한다.
**학습 전후 `val Top-1` 두 줄을 메모**해 둔다.

In [ ]:
assert AB_READY, "④ 준비가 안 됐어요. Drive에 yudam_colab_ab.zip을 올리고 1번 셀부터 다시 실행해 주세요"
%cd /content/yudam
!python train/train_metric.py --stage 4 --manifest train/manifest.csv --data-root data --out-dir /content/out --steps 300 --gallery-ids A01,B01 --hard-pairs train/hard_pairs_ab.csv
save_ckpt(4)

## 끝난 뒤
1. Drive `out/`의 `ckpt_stage4.pt`를 PC의 `models/` 폴더에 넣는다 (`ckpt_stage2.pt`·`ckpt_stage3.pt`는 덮어쓰지 않는다)
2. 학습 전후 val Top-1 두 줄을 알려 주면, PC에서 인덱스 재구축(`build_index.py 4`) → 임계값(val로만) → test 평가 → 비슷한 금동불상 구분 평가를 이어서 한다